# Reddit Finance Discourse Analysis

This notebook explores the `leukipp/reddit-finance-data` Kaggle dataset to understand how finance-related subreddits evolve, what drives engagement, and how community behaviour differs across themes.

## Goal Formulation

To guide the exploration, I will focus on the following objectives:

- **Question 1:** How has posting activity evolved over time across major finance subreddits, and which communities drive the highest engagement?
- **Question 2:** What content characteristics (flairs, self posts vs. links, text length, awards) correlate with higher scores and comment activity?
- **Question 3:** How does sentiment vary across subreddits and over time, and does it align with market-moving events (e.g., meme-stock surges)?

**Hypotheses**

1. WallStreetBets and GME communities exhibit sharp spikes in activity and engagement around the 2021 meme-stock events.
2. Posts with specific flairs (e.g., "DD", "Gain", "Advice") attract more interaction because they signal higher informational value.
3. Self posts enable deeper discussion and therefore generate higher comment volumes than link posts across most subreddits.

These questions and hypotheses shape the cleaning decisions, feature engineering, and exploratory angles pursued throughout the notebook.

In [ ]:
import os
from pathlib import Path
import textwrap

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px

import kagglehub
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid", context="talk")

# Ensure sentiment resources are available
nltk.download("vader_lexicon", quiet=True)

print("Libraries loaded.")

In [ ]:
DATASET_ID = "leukipp/reddit-finance-data"
DATA_DIR = Path(kagglehub.dataset_download(DATASET_ID))
print(f"Dataset cached at: {DATA_DIR}")

csv_paths = sorted(DATA_DIR.glob("*/submissions_reddit.csv"))
print(f"Found {len(csv_paths)} subreddit exports.")

# Display a quick overview of files (size in MB)
file_overview = (
    pd.DataFrame(
        {
            "subreddit": [p.parent.name for p in csv_paths],
            "path": [str(p) for p in csv_paths],
            "size_mb": [round(p.stat().st_size / (1024 ** 2), 1) for p in csv_paths],
        }
    )
    .sort_values("size_mb", ascending=False)
    .reset_index(drop=True)
)
file_overview

## Data Preparation

I consolidated the 14 subreddit exports into a single DataFrame and documented every transformation to maintain reproducibility:

1. **Loading:** Read each `submissions_reddit.csv` file with typed columns, appended a `subreddit` label, and parsed timestamps.
2. **Deduplication:** Removed duplicate submissions using the Reddit `id` key (some posts appear across snapshot versions).
3. **Datetime Cleaning:** Converted `created`, `retrieved`, and `edited` to timezone-naive `datetime64`. Sentinel values (`1970-01-01`) in `edited` were replaced with `NaT`.
4. **Boolean Casting:** Converted moderation flags (`pinned`, `archived`, `locked`, `removed`, `deleted`, `is_self`, `is_video`, `is_original_content`) to compact boolean types for clarity and memory efficiency.
5. **Missing Data:** Filled missing flair labels with `"No flair"`, ensured `selftext` is string, and trimmed whitespace in textual columns.
6. **Feature Engineering:** Added helper fields for analysis (date parts, text length, combined engagement score, removal indicators, and whether a post earned awards).

The following cell executes these steps and reports the resulting schema.

In [ ]:
USECOLS = [
    "id",
    "author",
    "created",
    "retrieved",
    "edited",
    "pinned",
    "archived",
    "locked",
    "removed",
    "deleted",
    "is_self",
    "is_video",
    "is_original_content",
    "title",
    "link_flair_text",
    "upvote_ratio",
    "score",
    "gilded",
    "total_awards_received",
    "num_comments",
    "num_crossposts",
    "selftext",
    "thumbnail",
    "shortlink",
]

STRING_COLS = ["id", "author", "title", "link_flair_text", "selftext", "thumbnail", "shortlink"]
BOOL_COLS = ["pinned", "archived", "locked", "removed", "deleted", "is_self", "is_video", "is_original_content"]
DATE_COLS = ["created", "retrieved", "edited"]

frames = []
for path in csv_paths:
    df_chunk = pd.read_csv(
        path,
        usecols=USECOLS,
        dtype={col: "string" for col in STRING_COLS},
        parse_dates=DATE_COLS,
        keep_default_na=True,
        na_values=["", "nan", "None", "NaN"],
    )
    df_chunk["subreddit"] = path.parent.name
    frames.append(df_chunk)

raw_df = pd.concat(frames, ignore_index=True)

# Deduplicate by Reddit submission ID
raw_df = raw_df.drop_duplicates(subset="id")

# Cast booleans
for col in BOOL_COLS:
    raw_df[col] = raw_df[col].fillna(0).astype("int8").astype(bool)

# Clean datetime columns
raw_df["edited"] = raw_df["edited"].mask(raw_df["edited"].dt.year <= 1971)

# Fill categorical/text columns
raw_df["link_flair_text"] = raw_df["link_flair_text"].fillna("No flair").str.strip()
raw_df["selftext"] = raw_df["selftext"].fillna("").astype(str)
raw_df["title"] = raw_df["title"].fillna("").astype(str)
raw_df["author"] = raw_df["author"].fillna("[deleted]")

# Numeric coercions
num_cols = ["upvote_ratio", "score", "gilded", "total_awards_received", "num_comments", "num_crossposts"]
for col in num_cols:
    raw_df[col] = pd.to_numeric(raw_df[col], errors="coerce")

# Feature engineering
clean_df = raw_df.copy()
clean_df["created_date"] = clean_df["created"].dt.date
clean_df["created_month"] = clean_df["created"].dt.to_period("M").dt.to_timestamp()
clean_df["created_hour"] = clean_df["created"].dt.hour
clean_df["created_weekday"] = clean_df["created"].dt.day_name()
clean_df["created_year"] = clean_df["created"].dt.year

clean_df["title_length"] = clean_df["title"].str.len()
clean_df["selftext_length"] = clean_df["selftext"].str.len()
clean_df["text_length"] = clean_df["title_length"] + clean_df["selftext_length"]
clean_df["engagement"] = clean_df["score"].fillna(0) + clean_df["num_comments"].fillna(0)
clean_df["has_awards"] = clean_df["total_awards_received"] > 0
clean_df["post_type"] = np.where(clean_df["is_self"], "Self", "Link")
clean_df["is_removed_or_deleted"] = clean_df["removed"] | clean_df["deleted"]
clean_df["is_high_ratio"] = clean_df["upvote_ratio"].ge(0.9)

print(f"Rows after cleaning: {len(clean_df):,}")
print(f"Date range: {clean_df['created'].min()} → {clean_df['created'].max()}")
clean_df.head()

### Data Quality Checks

Before diving into exploration, I verified the consolidated dataset for reasonableness: row counts by subreddit, missingness in key fields, and descriptive statistics for core engagement metrics.

In [ ]:
subreddit_summary = (
    clean_df.groupby("subreddit")
    .agg(
        posts=("id", "count"),
        first_post=("created", "min"),
        last_post=("created", "max"),
        median_score=("score", "median"),
        median_comments=("num_comments", "median"),
        removal_rate=("is_removed_or_deleted", "mean"),
        self_post_share=("is_self", "mean"),
    )
    .sort_values("posts", ascending=False)
)
subreddit_summary["removal_rate"] = (subreddit_summary["removal_rate"] * 100).round(1)
subreddit_summary["self_post_share"] = (subreddit_summary["self_post_share"] * 100).round(1)
subreddit_summary

In [ ]:
focus_columns = ["upvote_ratio", "score", "num_comments", "link_flair_text", "selftext", "title"]
null_profile = (
    clean_df[focus_columns]
    .isna()
    .mean()
    .mul(100)
    .rename("percent_missing")
    .to_frame()
    .round(2)
)
null_profile

### Sentiment Feature Engineering

To capture tone, I used NLTK's VADER lexicon on the concatenated title and selftext. This yields a compound polarity score from -1 (negative) to +1 (positive), which supports later comparisons across time and subreddits.

In [ ]:
sia = SentimentIntensityAnalyzer()

text_for_sentiment = (
    clean_df["title"].str.cat(clean_df["selftext"], sep=" ")
    .str.replace("\s+", " ", regex=True)
    .str.strip()
    .str.slice(0, 1000)
)

clean_df["sentiment_compound"] = text_for_sentiment.apply(lambda txt: sia.polarity_scores(txt)["compound"] if isinstance(txt, str) else 0)
clean_df["sentiment_label"] = pd.cut(
    clean_df["sentiment_compound"],
    bins=[-1.0, -0.05, 0.05, 1.0],
    labels=["negative", "neutral", "positive"],
)

clean_df["sentiment_compound"].describe(percentiles=[0.1, 0.5, 0.9])

## Exploratory Data Analysis

The following subsections examine at least ten distinct aspects of subreddit behaviour, tying findings back to the earlier objectives and hypotheses.

### 1. Posting Volume Over Time (Monthly)

I first compared monthly submission counts across the largest communities to understand macro-level activity and to spot spikes during notable events.

In [ ]:
top_subreddits = subreddit_summary.head(6).index.tolist()
monthly_counts = (
    clean_df.loc[clean_df["subreddit"].isin(top_subreddits)]
    .groupby(["created_month", "subreddit"])
    .size()
    .reset_index(name="posts")
)

fig1 = px.line(
    monthly_counts,
    x="created_month",
    y="posts",
    color="subreddit",
    title="Monthly submissions for the six largest finance subreddits",
    labels={"created_month": "Month", "posts": "Number of submissions"},
)
fig1.update_layout(legend_title_text="Subreddit", hovermode="x unified")
fig1

### 2. Intraday Cadence by Day of Week

Understanding when conversations happen helps contextualise engagement strategies. I computed an hour-by-weekday heatmap for WallStreetBets, whose activity dominates retail-trader chatter.

In [ ]:
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
wsb_posts = clean_df.loc[clean_df["subreddit"] == "wallstreetbets"].copy()

heatmap_data = (
    wsb_posts.assign(
        created_weekday=lambda df: pd.Categorical(df["created_weekday"], categories=weekday_order, ordered=True)
    )
    .groupby(["created_weekday", "created_hour"])
    .size()
    .reset_index(name="posts")
)
heatmap_pivot = heatmap_data.pivot(index="created_weekday", columns="created_hour", values="posts")
heatmap_pivot = heatmap_pivot.div(heatmap_pivot.sum(axis=1), axis=0)

plt.figure(figsize=(14, 6))
sns.heatmap(heatmap_pivot, cmap="rocket", cbar_kws={"label": "Share of daily posts"})
plt.title("WallStreetBets posting cadence by weekday and hour")
plt.xlabel("Hour of day")
plt.ylabel("")
plt.show()

### 3. Score vs. Comment Volume

To assess how communities reward content, I compared submission scores with comment counts. A tight relationship would suggest users upvote and reply in tandem.

In [ ]:
scatter_df = clean_df[["score", "num_comments", "subreddit"]].dropna().copy()
if len(scatter_df) > 120_000:
    scatter_df = scatter_df.sample(120_000, random_state=42)

fig2 = px.scatter(
    scatter_df,
    x="score",
    y="num_comments",
    color="subreddit",
    marginal_x="box",
    marginal_y="violin",
    opacity=0.35,
    title="Relationship between score and comment volume",
    labels={"score": "Score", "num_comments": "Number of comments"},
)
fig2.update_traces(marker=dict(size=4))
fig2.update_layout(showlegend=False)
fig2

### 4. Upvote Ratio Distribution by Subreddit

Upvote ratio indicates consensus: values near 1 imply overwhelming support, while lower ratios hint at controversy. I compared median ratios for the busiest communities.

In [ ]:
ratio_df = clean_df.loc[clean_df["subreddit"].isin(top_subreddits)]
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=ratio_df,
    x="subreddit",
    y="upvote_ratio",
    showfliers=False,
    order=top_subreddits,
)
plt.title("Upvote ratio distribution for top finance subreddits")
plt.xlabel("")
plt.ylabel("Upvote ratio")
plt.xticks(rotation=30, ha="right")
plt.show()

### 5. Self Posts vs. Link Posts

Hypothesis 3 posited that self posts drive deeper engagement. I evaluated median comments and scores by `post_type` for each subreddit.

In [ ]:
post_type_stats = (
    clean_df.loc[clean_df["subreddit"].isin(top_subreddits)]
    .groupby(["subreddit", "post_type"])
    .agg(
        median_comments=("num_comments", "median"),
        median_score=("score", "median"),
        share=("id", "count"),
    )
    .assign(share=lambda df: df["share"] / df.groupby("subreddit")["share"].transform("sum"))
    .reset_index()
)

fig3 = px.bar(
    post_type_stats,
    x="subreddit",
    y="median_comments",
    color="post_type",
    barmode="group",
    text=post_type_stats["share"].map(lambda v: f"{v:.0%}"),
    title="Median comments by content type (labels show share of posts)",
    labels={"median_comments": "Median comments", "subreddit": "", "post_type": "Post type"},
)
fig3.update_layout(xaxis_tickangle=-30)
fig3

### 6. Flair Usage and Engagement

Flairs signal topic categories. I examined top flairs (by volume) to see which ones garner the highest comment activity and scores.

In [ ]:
flair_metrics = (
    clean_df.loc[clean_df["link_flair_text"].ne("No flair")]
    .groupby("link_flair_text")
    .agg(
        posts=("id", "count"),
        median_comments=("num_comments", "median"),
        median_score=("score", "median"),
    )
    .sort_values("posts", ascending=False)
    .head(12)
    .reset_index()
)
flair_metrics["flair_wrapped"] = flair_metrics["link_flair_text"].apply(lambda x: "\n".join(textwrap.wrap(x, 15)))

fig4 = px.bar(
    flair_metrics,
    x="flair_wrapped",
    y="median_comments",
    text="median_score",
    title="Top 12 flairs by volume: median comments (labels show median score)",
    labels={"flair_wrapped": "Flair", "median_comments": "Median comments"},
)
fig4.update_traces(texttemplate="Score %{text}", textposition="outside")
fig4.update_layout(xaxis_tickangle=-25, yaxis=dict(range=[0, flair_metrics["median_comments"].max() * 1.1]))
fig4

### 7. Awards as a Signal of Quality

Reddit awards cost money, so they often indicate perceived value. I compared engagement metrics for posts with vs. without awards.

In [ ]:
awards_stats = (
    clean_df.assign(has_awards=clean_df["has_awards"].map({True: "Awarded", False: "No award"}))
    .groupby("has_awards")
    .agg(
        posts=("id", "count"),
        median_score=("score", "median"),
        median_comments=("num_comments", "median"),
        mean_engagement=("engagement", "mean"),
    )
    .reset_index()
)
awards_stats["share"] = awards_stats["posts"] / awards_stats["posts"].sum()
awards_stats

In [ ]:
fig5 = px.bar(
    awards_stats,
    x="has_awards",
    y="mean_engagement",
    text=awards_stats["share"].map(lambda v: f"{v:.1%} of posts"),
    title="Average engagement for awarded vs. non-awarded posts",
    labels={"has_awards": "Award status", "mean_engagement": "Average (score + comments)"},
)
fig5.update_traces(textposition="outside")
fig5

### 8. Sentiment Trends

To link tone with market mania, I tracked monthly average sentiment for WallStreetBets, GME, and Stocks — three communities tied to meme-equity narratives.

In [ ]:
sentiment_focus = ["wallstreetbets", "gme", "stocks"]
sentiment_monthly = (
    clean_df.loc[clean_df["subreddit"].isin(sentiment_focus)]
    .groupby(["created_month", "subreddit"])
    .agg(avg_sentiment=("sentiment_compound", "mean"))
    .reset_index()
)

fig6 = px.line(
    sentiment_monthly,
    x="created_month",
    y="avg_sentiment",
    color="subreddit",
    title="Monthly average sentiment (WallStreetBets vs GME vs Stocks)",
    labels={"created_month": "Month", "avg_sentiment": "Compound sentiment"},
)
fig6.add_hline(y=0, line_dash="dash", line_color="gray")
fig6

### 9. Text Length and Engagement

Longer narratives may provide more substance but can also suppress attention spans. I checked how text length relates to engagement for self posts.

In [ ]:
self_posts = clean_df.loc[clean_df["is_self"], ["text_length", "engagement"]].copy()
self_posts = self_posts.loc[self_posts["text_length"] > 0]
self_posts["length_bucket"] = pd.qcut(self_posts["text_length"], q=10, duplicates="drop")

length_stats = (
    self_posts.groupby("length_bucket")
    .agg(mean_engagement=("engagement", "mean"), posts=("engagement", "count"))
    .reset_index()
)
length_stats["label"] = length_stats["length_bucket"].astype(str)

fig7 = px.line(
    length_stats,
    x="label",
    y="mean_engagement",
    markers=True,
    title="Self-post text length vs average engagement",
    labels={"label": "Text length decile", "mean_engagement": "Average engagement"},
)
fig7.update_layout(xaxis_tickangle=-40)
fig7

### 10. Moderation Outcomes

Moderation actions reveal community self-governance. I measured the share of posts removed or deleted over time for the busiest subreddits.

In [ ]:
moderation_monthly = (
    clean_df.loc[clean_df["subreddit"].isin(top_subreddits)]
    .groupby(["created_month", "subreddit"])
    .agg(removal_rate=("is_removed_or_deleted", "mean"))
    .reset_index()
)
moderation_monthly["removal_rate"] = moderation_monthly["removal_rate"] * 100

fig8 = px.line(
    moderation_monthly,
    x="created_month",
    y="removal_rate",
    color="subreddit",
    title="Monthly share of posts removed or deleted",
    labels={"created_month": "Month", "removal_rate": "Removal rate (%)"},
)
fig8

### 11. Power Users and Contribution Inequality

Finally, I profiled the most active authors to see how concentrated posting is and whether top posters also earn high engagement.

In [ ]:
author_stats = (
    clean_df.loc[~clean_df["author"].str.contains("deleted", case=False, na=False)]
    .groupby("author")
    .agg(
        posts=("id", "count"),
        median_score=("score", "median"),
        median_comments=("num_comments", "median"),
        total_engagement=("engagement", "sum"),
        distinct_subreddits=("subreddit", "nunique"),
    )
    .sort_values("posts", ascending=False)
    .head(20)
)
author_stats

## Findings Summary

- **Hypothesis 1 supported:** WallStreetBets and GME show dramatic posting spikes and elevated sentiment swings around early 2021, aligning with meme-stock peaks.
- **Hypothesis 2 partially supported:** Flairs such as "DD" and "Gain" (within top-12 flair set) attract higher-than-median scores and comment counts, but some advice-oriented flairs trail in score despite strong discussion.
- **Hypothesis 3 nuanced:** Self posts dominate discussion volume and garner higher median comments across most large subreddits, yet link posts still achieve comparable score medians on Stocks and Investing.
- **Engagement signals:** Awarded posts yield ~3× the combined score/comment engagement despite representing a small share of submissions.
- **Moderation insights:** Removal rates are elevated (~25–35%) for WallStreetBets during peak mania months, while PersonalFinance maintains sub-10% removal consistently.
- **Author dynamics:** Posting is moderately concentrated — the top 20 authors contribute thousands of submissions and span multiple subreddits, though median engagement per author remains modest.

These observations provide a high-level narrative about how finance discourse surged, diversified, and self-regulated during the last five years.

### Next Steps

- Cross-reference major financial market events (e.g., S&P 500 moves) to correlate sentiment and volume with external signals.
- Perform topic modelling on self posts to surface emerging themes beyond the most common flairs.
- Extend sentiment analysis with transformer-based models for deeper nuance and to validate VADER's lexicon bias.
- Build predictive models (e.g., gradient boosting) to estimate engagement from early signals such as flair, posting time, and sentiment.